# Task 2: Лаборатори 7,8,9 — Цуйван хийх аргыг өөрийн хүссэн гаралт гаргадаг болгох

**Model:** `Qwen/Qwen2.5-3B-Instruct`
**Техник:** QLoRA (4-bit quantization + LoRA adapter)
**Dataset:** `data/tsuivan_dataset.json` (30 жишээ, Alpaca формат)
**Гаралтын формат:** Орц → Алхам → Зөвлөгөө
**Hardware:** RTX 5080 (16GB VRAM)

In [ ]:
#!pip install -q -U transformers peft bitsandbytes accelerate trl datasets

In [1]:
import json
import torch
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
DATASET_PATH = "data/tsuivan_dataset.json"
OUTPUT_DIR = "./qwen-tsuivan-lora"

/home/toru2/Amara/Deep_learning/midterm2/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model)

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]/home/toru2/Amara/Deep_learning/midterm2/venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 434/434 [00:00<00:00, 865.53it/s]


In [3]:
with open(DATASET_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

def format_chat(example):
    messages = [
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["output"]},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return {"text": text}

dataset = Dataset.from_list(raw_data).map(format_chat)
print(f"Нийт жишээ: {len(dataset)}")
print("--- Жишээ ---")
print(dataset[0]["text"][:500])

Map: 100%|██████████| 30/30 [00:00<00:00, 3662.72 examples/s]

Нийт жишээ: 30
--- Жишээ ---
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Цуйван хэрхэн хийх вэ?<|im_end|>
<|im_start|>assistant
Орц:
- Гурил: 500г
- Үхрийн мах: 300г
- Төмс: 2 ширхэг
- Лууван: 1 ширхэг
- Байцаа: 200г
- Сонгино: 1 ширхэг
- Сармис: 3 хумс
- Ургамлын тос: 3 халбага
- Давс, хар перц: амтлах
- Соёо сойвой: 2 халбага

Алхам:
1. Гурилд давс, ус нэмж хатуувтар зуурна. 20 минут амраана.
2. Зуурмагаа нимгэн дэлгэж, уураар 15 минут жигнэнэ.
3. Жигн


In [4]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607


In [17]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=10,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    weight_decay=0.01,
    max_grad_norm=0.3,
    optim="paged_adamw_8bit",
    logging_steps=5,
    save_strategy="epoch",
    bf16=True,
    report_to="none",
    # group_by_length=True,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    # tokenizer=tokenizer,
    args=training_args,
    # dataset_text_field="text",
    # max_seq_length=1024,
    # packing=False,
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
Tokenizing train dataset: 100%|██████████| 30/30 [00:00<00:00, 2174.04 examples/s]


In [18]:
trainer.train()

/home/toru2/Amara/Deep_learning/midterm2/venv/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
5,1.216723
10,0.957883
15,0.740043
20,0.539394
25,0.418048
30,0.285317
35,0.237046
40,0.231853


/home/toru2/Amara/Deep_learning/midterm2/venv/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/home/toru2/Amara/Deep_learning/midterm2/venv/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences bet

TrainOutput(global_step=40, training_loss=0.5782882392406463, metrics={'train_runtime': 67.0611, 'train_samples_per_second': 4.474, 'train_steps_per_second': 0.596, 'total_flos': 2169429899378688.0, 'train_loss': 0.5782882392406463})

In [19]:
trainer.model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Загвар хадгалагдлаа: {OUTPUT_DIR}")

Загвар хадгалагдлаа: ./qwen-tsuivan-lora


In [20]:
model.config.use_cache = True
model.eval()

def ask(question, max_new_tokens=512):
    messages = [{"role": "user", "content": question}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    answer = tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return answer

print(ask("Цуйван хэрхэн хийх вэ?"))

/home/toru2/Amara/Deep_learning/midterm2/venv/lib/python3.12/site-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Орц:
- Гурил: 500г
- Мах: 300г (цагаан га)
- Төмс: 2 ширхэг
- Лууван: 1 ширхэг (цагаан га)
- Байцаа: 200г
- Сонгино: 1 ширхэг
- Сармис: 4 хумс
- Давс, перц, соёо сойвой

Алхам:
1. Гурилаа зуурч нимгэн дэлгэж жигнэнэ.
2. Нимгэн гурилаа халуун тосонд хуурна.
3. Халуун тос дараа махаа нэмж хуурна.
4. Цагаан гаа дэлгэж амтална.
5. Төмс, лууван, сонгиноо нэмж хайрна.
6. Байцаагаа нэмж хуурна.
7. Давс, соёо сойвойгоор амтална.
8. Гурилаа нэмж хольж 10 минут болгоно.

Зөвлөгөө:
- Цагаан гаа хурдан болдог гиш нь хэт хуурахад амтыг баялаг болгоно.
- Бага халааж мах сайтар хуурна.
- Тахиас амталնays гиш нь хүнд гаргадвал ч хатуу болно.


In [21]:
test_questions = [
    "Үхрийн махтай цуйван хийх жор хэлнэ үү.",
    "Махгүй цуйван хэрхэн хийх вэ?",
    "Цуйванд ямар хүнсний ногоо ордог вэ?",
    "Цуйван хурдан хийх арга хэлнэ үү.",
]

for q in test_questions:
    print(f"\n{'='*60}\nАсуулт: {q}\n{'='*60}")
    print(ask(q))


Асуулт: Үхрийн махтай цуйван хийх жор хэлнэ үү.
Орц:
- Гурил: 500г
- Мах: 300г (үхрийн)
- Төмс: 2 ширхэг
- Лууван: 1 ширхэг
- Байцаа: 200г
- Сонгино: 1 ширхэг
- Сармис: 4 хумс
- Чинжүү: 1 халбага
- Давс, хар перц: амтлах
- Ургамлын тос: 2 халбага

Алхам:
1. Гурилаа зуурч нимгэн дэлгэж 20 минут амраана.
2. үхрийн махаа хайрна.
3. Халуун тосонд сонгино, сармисаа хуурна.
4. Махаа гурилаааааa нэмнэ.
5. Төмс, лууванаа нэмж хольно.
6. Байцаагаа нэмж 5 минут хуурна.
7. Сармис, перц, давсаа амтална.
8. Гурилаа нь хооллana.

Зөвлөгөө:
- Үхрийн мах нь хүнсний ногоог нимгэн дээр зуурна.
- Махны зүсэх нь бага эвдрэг болгоно.
- Сармис, перц ширээд жаахан болгоно.

Асуулт: Махгүй цуйван хэрхэн хийх вэ?
Орц:
- Гурил: 500г
- Мөөг: 300г
- Лууван: 1 ширхэг (заавал)
- Байцаа: 200г (заавал)
- Төмс: 2 ширхэг
- Сонгино: 1 ширхэг (заавал)
- Сармис: 3 хумс
- Давс, перц, соёо сойвой

Алхам:
1. Гурилаа зууран амраана.
2. Нимгэн дэлгэж уураар жигнэнэ.
3. Жигнэсэн гурилаа нарийн зүснэ.
4. Мөөгөө нэмж 3 минут хуу

## Дүгнэлт

- Qwen2.5-3B-Instruct моделийг QLoRA аргаар цуйваны 30 жишээн дээр fine-tune хийв.
- Гаралт нь "Орц → Алхам → Зөвлөгөө" форматтай болж тогтсон.
- LoRA adapter зөвхөн ~1% параметр сургаж RTX 5080 (16GB)-д багтсан.
- Илүү олон жишээ (100+) нэмэх, epoch тохируулах, DPO ашиглан илүү сайжруулж болно.